In [1]:
import pandas as pd
import numpy as np
import sqlite3
import os
import warnings
warnings.filterwarnings('ignore')

print("All imports successful")

All imports successful


In [2]:
transaction = pd.read_csv('../data/raw/train_transaction.csv')
identity = pd.read_csv('../data/raw/train_identity.csv')

df = transaction.merge(identity, on='TransactionID', how='left')

print(f"Merged shape: {df.shape}")
print("Data loaded successfully")

Merged shape: (590540, 434)
Data loaded successfully


In [3]:
# Separate numeric and categorical columns
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()

# Fill numeric nulls with median
df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].median())

# Fill categorical nulls with 'unknown'
df[categorical_cols] = df[categorical_cols].fillna('unknown')

# Verify no nulls remain
remaining_nulls = df.isnull().sum().sum()
print(f"Numeric columns   : {len(numeric_cols)}")
print(f"Categorical columns: {len(categorical_cols)}")
print(f"Remaining nulls   : {remaining_nulls}")
print("Cleaning complete")

Numeric columns   : 403
Categorical columns: 31
Remaining nulls   : 0
Cleaning complete


In [4]:
# Feature 1: Transaction hour
df['hour'] = (df['TransactionDT'] // 3600) % 24

# Feature 2: Transaction day
df['day'] = (df['TransactionDT'] // (3600 * 24)) % 7

# Feature 3: Transaction amount z-score
df['amt_zscore'] = (df['TransactionAmt'] - df['TransactionAmt'].mean()) / df['TransactionAmt'].std()

# Feature 4: Log transaction amount
df['amt_log'] = np.log1p(df['TransactionAmt'])

# Feature 5: High amount flag (top 10%)
amt_90 = df['TransactionAmt'].quantile(0.90)
df['high_amt_flag'] = (df['TransactionAmt'] > amt_90).astype(int)

# Feature 6: Card velocity — transaction count per card
card_velocity = df.groupby('card1')['TransactionID'].transform('count')
df['card_velocity'] = card_velocity

print(f"New features added: hour, day, amt_zscore, amt_log, high_amt_flag, card_velocity")
print(f"New shape: {df.shape}")
print("Feature engineering complete")

New features added: hour, day, amt_zscore, amt_log, high_amt_flag, card_velocity
New shape: (590540, 440)
Feature engineering complete


In [5]:
# Dimension 1: Card dimension
dim_card = df[['card1', 'card2', 'card3', 'card4', 'card5', 'card6']].drop_duplicates().reset_index(drop=True)
dim_card['card_id'] = dim_card.index + 1

# Dimension 2: Date dimension
dim_date = df[['hour', 'day']].drop_duplicates().reset_index(drop=True)
dim_date['date_id'] = dim_date.index + 1

# Dimension 3: Email dimension
dim_email = df[['P_emaildomain', 'R_emaildomain']].drop_duplicates().reset_index(drop=True)
dim_email['email_id'] = dim_email.index + 1

# Dimension 4: Device dimension
dim_device = df[['DeviceType', 'DeviceInfo']].drop_duplicates().reset_index(drop=True)
dim_device['device_id'] = dim_device.index + 1

print(f"dim_card rows   : {len(dim_card):,}")
print(f"dim_date rows   : {len(dim_date):,}")
print(f"dim_email rows  : {len(dim_email):,}")
print(f"dim_device rows : {len(dim_device):,}")
print("Dimension tables created")

dim_card rows   : 14,885
dim_date rows   : 168
dim_email rows  : 743
dim_device rows : 1,943
Dimension tables created


In [6]:
# Merge dimension keys back to main df
df = df.merge(dim_card[['card1', 'card2', 'card3', 'card4', 'card5', 'card6', 'card_id']], 
              on=['card1', 'card2', 'card3', 'card4', 'card5', 'card6'], how='left')

df = df.merge(dim_date[['hour', 'day', 'date_id']], 
              on=['hour', 'day'], how='left')

df = df.merge(dim_email[['P_emaildomain', 'R_emaildomain', 'email_id']], 
              on=['P_emaildomain', 'R_emaildomain'], how='left')

df = df.merge(dim_device[['DeviceType', 'DeviceInfo', 'device_id']], 
              on=['DeviceType', 'DeviceInfo'], how='left')

# Build fact table with key columns
fact_transactions = df[[
    'TransactionID', 'TransactionDT', 'TransactionAmt',
    'isFraud', 'card_id', 'date_id', 'email_id', 'device_id',
    'amt_zscore', 'amt_log', 'high_amt_flag', 'card_velocity',
    'hour', 'day', 'card4', 'card6'
]].copy()

print(f"Fact table shape: {fact_transactions.shape}")
print(f"Fraud count in fact table: {fact_transactions['isFraud'].sum():,}")
print("Fact table created")


Fact table shape: (590540, 16)
Fraud count in fact table: 20,663
Fact table created


In [7]:
# Create database
db_path = '../data/fraud_detection.db'
conn = sqlite3.connect(db_path)

# Write all tables
fact_transactions.to_sql('fact_transactions', conn, if_exists='replace', index=False)
dim_card.to_sql('dim_card', conn, if_exists='replace', index=False)
dim_date.to_sql('dim_date', conn, if_exists='replace', index=False)
dim_email.to_sql('dim_email', conn, if_exists='replace', index=False)
dim_device.to_sql('dim_device', conn, if_exists='replace', index=False)

conn.close()

# Verify file created
file_size = os.path.getsize(db_path) / (1024 * 1024)
print(f"Database created : fraud_detection.db")
print(f"File size        : {file_size:.1f} MB")
print(f"Tables loaded    : fact_transactions, dim_card, dim_date, dim_email, dim_device")
print("SQLite load complete")

Database created : fraud_detection.db
File size        : 39.9 MB
Tables loaded    : fact_transactions, dim_card, dim_date, dim_email, dim_device
SQLite load complete


In [8]:
conn = sqlite3.connect('../data/fraud_detection.db')

# Verify all tables
tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)
print("Tables in database:")
print(tables.to_string(index=False))

print()

# Verify fact table
fact_check = pd.read_sql("SELECT COUNT(*) as rows, SUM(isFraud) as fraud_count, ROUND(AVG(isFraud)*100, 2) as fraud_rate FROM fact_transactions", conn)
print("Fact table verification:")
print(fact_check.to_string(index=False))

print()

# Verify joins work
join_check = pd.read_sql("""
    SELECT 
        dc.card4,
        COUNT(*) as transactions,
        ROUND(AVG(ft.isFraud)*100, 2) as fraud_rate
    FROM fact_transactions ft
    JOIN dim_card dc ON ft.card_id = dc.card_id
    GROUP BY dc.card4
    ORDER BY fraud_rate DESC
""", conn)
print("Star schema join test — fraud rate by card network:")
print(join_check.to_string(index=False))

conn.close()

Tables in database:
             name
fact_transactions
         dim_card
         dim_date
        dim_email
       dim_device

Fact table verification:
  rows  fraud_count  fraud_rate
590540        20663         3.5

Star schema join test — fraud rate by card network:
           card4  transactions  fraud_rate
        discover          6651        7.73
            visa        384767        3.48
      mastercard        189217        3.43
american express          8328        2.87
         unknown          1577        2.60


In [9]:
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import LabelEncoder

# Prepare features for SMOTE
smote_features = ['TransactionAmt', 'amt_zscore', 'amt_log', 
                  'high_amt_flag', 'card_velocity', 'hour', 'day']

X = fact_transactions[smote_features]
y = fact_transactions['isFraud']

print(f"Before SMOTE:")
print(f"  Legitimate : {(y==0).sum():,}")
print(f"  Fraudulent : {(y==1).sum():,}")
print(f"  Ratio      : {(y==0).sum() / (y==1).sum():.1f}:1")

# Apply SMOTE
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X, y)

print(f"\nAfter SMOTE:")
print(f"  Legitimate : {(y_resampled==0).sum():,}")
print(f"  Fraudulent : {(y_resampled==1).sum():,}")
print(f"  Ratio      : {(y_resampled==0).sum() / (y_resampled==1).sum():.1f}:1")
print("SMOTE complete")

Before SMOTE:
  Legitimate : 569,877
  Fraudulent : 20,663
  Ratio      : 27.6:1

After SMOTE:
  Legitimate : 569,877
  Fraudulent : 569,877
  Ratio      : 1.0:1
SMOTE complete


In [10]:
import pickle

# Save resampled data for Phase 4 model
smote_data = {'X_resampled': X_resampled, 'y_resampled': y_resampled}
with open('../data/smote_resampled.pkl', 'wb') as f:
    pickle.dump(smote_data, f)

print("=" * 55)
print("PHASE 2 ETL PIPELINE — SUMMARY")
print("=" * 55)
print(f"Records processed     : {len(df):,}")
print(f"Numeric nulls filled  : median imputation")
print(f"Categorical nulls     : filled with 'unknown'")
print(f"Remaining nulls       : 0")
print()
print("New features engineered:")
print("  - hour            : transaction hour of day")
print("  - day             : transaction day of week")
print("  - amt_zscore      : amount z-score")
print("  - amt_log         : log transformed amount")
print("  - high_amt_flag   : top 10% amount indicator")
print("  - card_velocity   : transaction count per card")
print()
print("Star schema tables:")
print(f"  - fact_transactions : {len(fact_transactions):,} rows")
print(f"  - dim_card          : {len(dim_card):,} rows")
print(f"  - dim_date          : {len(dim_date):,} rows")
print(f"  - dim_email         : {len(dim_email):,} rows")
print(f"  - dim_device        : {len(dim_device):,} rows")
print()
print("SMOTE balancing:")
print(f"  Before : 27.6:1 imbalance")
print(f"  After  : 1:1 balanced")
print()
print("Output files:")
print("  - fraud_detection.db")
print("  - smote_resampled.pkl")
print("=" * 55)
print("Phase 2 complete. Notebook: 02_etl_pipeline.ipynb")

PHASE 2 ETL PIPELINE — SUMMARY
Records processed     : 590,540
Numeric nulls filled  : median imputation
Categorical nulls     : filled with 'unknown'
Remaining nulls       : 0

New features engineered:
  - hour            : transaction hour of day
  - day             : transaction day of week
  - amt_zscore      : amount z-score
  - amt_log         : log transformed amount
  - high_amt_flag   : top 10% amount indicator
  - card_velocity   : transaction count per card

Star schema tables:
  - fact_transactions : 590,540 rows
  - dim_card          : 14,885 rows
  - dim_date          : 168 rows
  - dim_email         : 743 rows
  - dim_device        : 1,943 rows

SMOTE balancing:
  Before : 27.6:1 imbalance
  After  : 1:1 balanced

Output files:
  - fraud_detection.db
  - smote_resampled.pkl
Phase 2 complete. Notebook: 02_etl_pipeline.ipynb
